# Day 13 — Machine-learning forecasts of S&P 500 return magnitude
**Question:** Can richer predictors, regularization, or nonlinear relationships improve next-trading-day absolute log-return forecasts?

Training outcomes end in 2022. Five expanding-window training folds select tuning parameters. Fixed fitted models then forecast 2023–2024 using predictors updated at each forecast date. **2025 is excluded before feature construction and remains untouched for final testing.**

Absolute returns are a noisy proxy for volatility. This is a magnitude-forecasting exercise, not an evaluation of trading profit.


## 1. Install packages

In [ ]:
%pip install pandas numpy matplotlib scikit-learn


## 2. Imports and project paths


In [1]:
from pathlib import Path
import hashlib
import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
try:
    from IPython.display import display
except ImportError:
    display = print
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

PROJECT_ROOT = Path('/Users/yaasarpong/Desktop/PHD/Year 4 fall/finance-volatility')
DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
FIGURES_DIR = PROJECT_ROOT / 'figures'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
file_path = DATA_DIR / 'financial_raw_data.csv'
assert file_path.exists(), f'Missing input: {file_path}'

def save_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / filename, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)


## 3. Build predictors on the full S&P trading sequence


In [2]:
data = pd.read_csv(file_path, parse_dates=['date'])
assert {'date', 'sp500', 'vix', 'treasury_10yr'}.issubset(data.columns)
assert data['date'].notna().all(), 'Missing/invalid dates.'
assert not data['date'].duplicated().any(), 'Duplicate dates.'
data = data.loc[data['date'] <= pd.Timestamp('2024-12-31')].sort_values('date').reset_index(drop=True)
base_columns = ['sp500', 'vix', 'treasury_10yr']
for col in base_columns:
    data[col] = pd.to_numeric(data[col], errors='raise')
assert not np.isinf(data[base_columns].to_numpy()).any()
data = data.dropna(subset=['sp500']).reset_index(drop=True)
assert data['sp500'].gt(0).all()

data['log_return'] = 100 * np.log(data['sp500']).diff()
data['absolute_return'] = data['log_return'].abs()
data['squared_return'] = data['log_return'] ** 2
data['return_lag_1'] = data['log_return'].shift(1)
data['abs_lag_1'] = data['absolute_return'].shift(1)
data['abs_lag_5'] = data['absolute_return'].shift(5)
for window in [5, 20]:
    data[f'mean_abs_{window}'] = data['absolute_return'].rolling(window, min_periods=window).mean()
    data[f'return_std_{window}'] = data['log_return'].rolling(window, min_periods=window).std(ddof=1)
data['treasury_change'] = data['treasury_10yr'].diff()
data['target_next_day'] = data['absolute_return'].shift(-1)
data['target_date'] = data['date'].shift(-1)
data['historical_mean'] = data['absolute_return'].expanding(min_periods=20).mean()
data['ewm_20'] = data['absolute_return'].ewm(span=20, adjust=False, min_periods=20).mean()

day12_features = ['absolute_return', 'mean_abs_5', 'mean_abs_20', 'vix', 'treasury_change']
ml_features = day12_features + [
    'log_return', 'return_lag_1', 'abs_lag_1', 'abs_lag_5',
    'squared_return', 'return_std_5', 'return_std_20',
]
baseline_columns = ['historical_mean', 'ewm_20']

definitions = {
    'absolute_return': ('Absolute log return today: |100 log(P_t/P_t-1)|', 'percentage points'),
    'mean_abs_5': ('Mean absolute return over 5 trading observations ending today', 'percentage points'),
    'mean_abs_20': ('Mean absolute return over 20 trading observations ending today', 'percentage points'),
    'vix': ("Today's VIX closing level", 'VIX index points'),
    'treasury_change': ("Today's 10-year yield minus yield at previous S&P observation; missing if either is missing", 'yield percentage points'),
    'log_return': ('Signed log return today', 'percentage points'),
    'return_lag_1': ('Signed log return at previous trading observation', 'percentage points'),
    'abs_lag_1': ('Absolute return at previous trading observation', 'percentage points'),
    'abs_lag_5': ('Absolute return 5 trading observations before today', 'percentage points'),
    'squared_return': ("Square of today's percentage-point log return", 'squared percentage points'),
    'return_std_5': ('Sample standard deviation of signed returns over 5 trading observations ending today', 'percentage points; not annualized'),
    'return_std_20': ('Sample standard deviation of signed returns over 20 trading observations ending today', 'percentage points; not annualized'),
}
feature_dictionary = pd.DataFrame([
    {'feature': name, 'definition': definitions[name][0], 'units': definitions[name][1]}
    for name in ml_features
])
display(feature_dictionary)
display(data[['date', 'absolute_return', 'target_date', 'target_next_day']].head())


,feature,definition,units
0,absolute_return,Absolute log return today: |100 log(P_t/P_t-1)|,percentage points
1,mean_abs_5,Mean absolute return over 5 trading observatio...,percentage points
2,mean_abs_20,Mean absolute return over 20 trading observati...,percentage points
3,vix,Today's VIX closing level,VIX index points
4,treasury_change,Today's 10-year yield minus yield at previous ...,yield percentage points
5,log_return,Signed log return today,percentage points
6,return_lag_1,Signed log return at previous trading observation,percentage points
7,abs_lag_1,Absolute return at previous trading observation,percentage points
8,abs_lag_5,Absolute return 5 trading observations before ...,percentage points
9,squared_return,Square of today's percentage-point log return,squared percentage points


,date,absolute_return,target_date,target_next_day
0,2016-10-03,NaN,2016-10-04,0.496790
1,2016-10-04,0.496790,2016-10-05,0.428749
2,2016-10-05,0.428749,2016-10-06,0.048143
3,2016-10-06,0.048143,2016-10-07,0.325877
4,2016-10-07,0.325877,2016-10-10,0.459537


## 4. Split by outcome date and align the evaluation sample
The new models use identical training rows and identical validation rows. The Day 12 reference is refitted using its original feature set and its original eligible pre-2023 training rows. This preserves its training definition; every method is then scored on the **shared Day 13 validation dates**. Same input data are required to reproduce the earlier model exactly.


In [3]:
cutoff = pd.Timestamp('2022-12-31')
validation_start = pd.Timestamp('2023-01-01')
validation_end = pd.Timestamp('2024-12-31')
day12_usable = data.dropna(subset=day12_features + baseline_columns + ['target_date', 'target_next_day']).copy()
day12_train = day12_usable.loc[day12_usable['target_date'] <= cutoff].copy()
usable = data.dropna(subset=ml_features + baseline_columns + ['target_date', 'target_next_day']).copy()
train = usable.loc[usable['target_date'] <= cutoff].copy()
validation = usable.loc[usable['target_date'].between(validation_start, validation_end)].copy()
assert len(train) > 800, 'Too few training rows for the selected fold sizes.'
assert not validation.empty
for frame in [train, validation]:
    assert (frame['date'] < frame['target_date']).all()
    assert np.isfinite(frame[ml_features + baseline_columns + ['target_next_day']].to_numpy()).all()
assert train['target_date'].max() <= validation['date'].min()

X_train, y_train = train[ml_features], train['target_next_day']
X_validation = validation[ml_features]
for name, frame in [('Day 12 reference training', day12_train), ('ML training', train), ('Shared validation', validation)]:
    print(name, '| Rows:', len(frame), '| Outcome dates:',
          frame['target_date'].min().date(), 'to', frame['target_date'].max().date())


Day 12 reference training | Rows: 1528 | Outcome dates: 2016-11-01 to 2022-12-30
ML training | Rows: 1528 | Outcome dates: 2016-11-01 to 2022-12-30
Shared validation | Rows: 496 | Outcome dates: 2023-01-03 to 2024-12-31


## 5. Define expanding-window tuning folds
Only outcomes through 2022 participate. Each validation fold contains 126 retained observations (roughly six trading months, with gaps from missing inputs). Calendar durations can differ; inspect printed dates. A one-row gap conservatively separates the training labels from the next fold's first forecast origin. The timestamp assertion checks this directly.

`TimeSeriesSplit` maintains order and grows the training window. See [official documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html).


In [4]:
splitter = TimeSeriesSplit(n_splits=5, test_size=126, gap=1)
cv_splits = list(splitter.split(X_train))
fold_rows = []
for number, (fit_idx, tune_idx) in enumerate(cv_splits, start=1):
    fit_rows, tune_rows = train.iloc[fit_idx], train.iloc[tune_idx]
    assert fit_rows['target_date'].max() < tune_rows['date'].min()
    fold_rows.append({
        'fold': number,
        'training_rows': len(fit_rows),
        'training_outcome_end': fit_rows['target_date'].max(),
        'tuning_origin_start': tune_rows['date'].min(),
        'tuning_outcome_start': tune_rows['target_date'].min(),
        'tuning_outcome_end': tune_rows['target_date'].max(),
        'tuning_rows': len(tune_rows),
    })
fold_table = pd.DataFrame(fold_rows)
display(fold_table)


,fold,training_rows,training_outcome_end,tuning_origin_start,tuning_outcome_start,tuning_outcome_end,tuning_rows
0,1,897,2020-06-12,2020-06-15,2020-06-16,2020-12-17,126
1,2,1023,2020-12-16,2020-12-17,2020-12-18,2021-06-21,126
2,3,1149,2021-06-18,2021-06-21,2021-06-22,2021-12-23,126
3,4,1275,2021-12-22,2021-12-23,2021-12-27,2022-06-27,126
4,5,1401,2022-06-24,2022-06-27,2022-06-28,2022-12-30,126


## 6. Fit the reference models and tune ridge, lasso, and random forest
**Primary tuning criterion: RMSE**, chosen before examining Day 13 validation results. MAE and bias remain secondary evaluation measures. RMSE aligns tuning with the squared-error fitting objective; MAE and RMSE may rank methods differently.

Scaling sits inside each pipeline, so means and standard deviations are fitted only on that fold's training rows. Random forest needs no scaling. `GridSearchCV` refits the winning specification on the full pre-2023 ML training sample. See [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) and [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html).

Larger alpha means stronger regularization within a given model. Ridge shrinks coefficients; lasso can set them to zero. The small forest grid limits depth and uses larger leaves to control complexity. Tuning may take a few minutes. If convergence warnings appear, address them before interpreting lasso results; do not suppress them.


In [5]:
def scaled_model(estimator):
    return Pipeline([('scaler', StandardScaler()), ('model', estimator)])

models = {}
day12_reference = LinearRegression().fit(day12_train[day12_features], day12_train['target_next_day'])
models['linear_expanded'] = scaled_model(LinearRegression()).fit(X_train, y_train)

specifications = {
    'ridge': (scaled_model(Ridge()), {'model__alpha': [0.01, 0.1, 1, 10, 100, 1000]}),
    'lasso': (scaled_model(Lasso(max_iter=50000, tol=1e-5)),
              {'model__alpha': [0.0001, 0.001, 0.01, 0.05, 0.1]}),
    'random_forest': (RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=1),
                      {'max_depth': [3, 6], 'min_samples_leaf': [10, 30], 'max_features': [0.7, 1.0]}),
}
searches = {}
tuning_rows = []
for name, (estimator, grid) in specifications.items():
    print('Tuning:', name, flush=True)
    search = GridSearchCV(estimator, grid, cv=cv_splits,
                          scoring='neg_root_mean_squared_error',
                          refit=True, n_jobs=2, error_score='raise')
    search.fit(X_train, y_train)
    searches[name] = search
    models[name] = search.best_estimator_
    tuning_rows.append({'method': name, 'mean_fold_RMSE': -search.best_score_,
                        'best_parameters': json.dumps(search.best_params_)})
    print('Best parameters:', search.best_params_)
tuning_summary = pd.DataFrame(tuning_rows)
display(tuning_summary)


Tuning: ridge
Best parameters: {'model__alpha': 1000}
Tuning: lasso
Best parameters: {'model__alpha': 0.05}
Tuning: random_forest
Best parameters: {'max_depth': 3, 'max_features': 0.7, 'min_samples_leaf': 10}


,method,mean_fold_RMSE,best_parameters
0,ridge,0.711184,"{""model__alpha"": 1000}"
1,lasso,0.721065,"{""model__alpha"": 0.05}"
2,random_forest,0.732943,"{""max_depth"": 3, ""max_features"": 0.7, ""min_sam..."


## 7. Generate and evaluate fixed-model forecasts
Every method predicts exactly the same outcome dates. Bias is forecast minus actual: positive means overprediction. All errors are in percentage points. Negative forecasts are counted and scored unchanged; clipping would be a separate specification.

The Day 12 reference uses fewer predictors than the four new models. Comparisons among the four new models isolate differences in fitting method for the same feature set; comparison with Day 12 also changes the predictors.


In [6]:
predictions = validation[['date', 'target_date', 'target_next_day', *baseline_columns]].copy()
predictions['day12_extended'] = day12_reference.predict(validation[day12_features])
for name, model in models.items():
    predictions[name] = model.predict(X_validation)
methods = baseline_columns + ['day12_extended'] + list(models)
assert np.isfinite(predictions[methods].to_numpy()).all()

score_rows = []
for name in methods:
    error = predictions[name] - predictions['target_next_day']
    score_rows.append({'method': name, 'n_forecasts': len(error),
                       'MAE': error.abs().mean(), 'RMSE': np.sqrt((error ** 2).mean()),
                       'Bias': error.mean(), 'negative_forecasts': int((predictions[name] < 0).sum())})
performance = pd.DataFrame(score_rows).set_index('method').sort_values('RMSE')
for reference in ['ewm_20', 'day12_extended']:
    for metric in ['MAE', 'RMSE']:
        performance[f'{metric}_improvement_vs_{reference}_pct'] = 100 * (
            1 - performance[metric] / performance.loc[reference, metric])
display(performance.round(4))
print('Lowest validation RMSE:', performance['RMSE'].idxmin())
print('Lowest validation MAE:', performance['MAE'].idxmin())
print('Positive improvement percentages mean lower error than the reference.')


,n_forecasts,MAE,RMSE,Bias,negative_forecasts,MAE_improvement_vs_ewm_20_pct,RMSE_improvement_vs_ewm_20_pct,MAE_improvement_vs_day12_extended_pct,RMSE_improvement_vs_day12_extended_pct
method,,,,,,,,,
random_forest,496,0.4065,0.5180,-0.0416,0,3.8887,2.2849,2.1272,1.5434
lasso,496,0.4127,0.5182,-0.0113,0,2.4174,2.2420,0.6290,1.5001
ridge,496,0.4189,0.5218,0.0165,0,0.9583,1.5731,-0.8569,0.8261
day12_extended,496,0.4153,0.5261,-0.0325,0,1.7998,0.7532,0.0000,0.0000
linear_expanded,496,0.4159,0.5281,-0.0332,0,1.6531,0.3820,-0.1494,-0.3741
ewm_20,496,0.4229,0.5301,0.0037,0,0.0000,0.0000,-1.8328,-0.7589
historical_mean,496,0.4591,0.5471,0.1425,0,-8.5576,-3.2057,-10.5472,-3.9890


Lowest validation RMSE: random_forest
Lowest validation MAE: random_forest
Positive improvement percentages mean lower error than the reference.


## 8. Plot and save reproducibility outputs
The score chart compares all methods; the forecast chart compares EWM, Day 12, and random forest with the realized target. Standardized coefficients are descriptive and are not causal effects. Lasso may choose one of several correlated features, so nonzero coefficients are not definitive importance rankings.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, metric in zip(axes, ['MAE', 'RMSE']):
    performance[metric].sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'2023–2024 validation {metric}: lower is better')
    ax.set_xlabel('Percentage points')
    ax.set_ylabel('')
save_figure(fig, 'day13_model_comparison.png')

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(predictions['target_date'], predictions['target_next_day'], color='gray',
        linewidth=0.7, alpha=0.5, label='Actual absolute return')
for name in ['ewm_20', 'day12_extended', 'random_forest']:
    ax.plot(predictions['target_date'], predictions[name], linewidth=1, label=name)
ax.set_xlabel('Outcome date')
ax.set_ylabel('Absolute log return (%)')
ax.set_title('Shared validation dates: fixed fitted models')
ax.legend()
save_figure(fig, 'day13_validation_forecasts.png')

coefficient_table = pd.DataFrame({
    name: models[name].named_steps['model'].coef_
    for name in ['linear_expanded', 'ridge', 'lasso']
}, index=ml_features).rename_axis('feature')
display(coefficient_table.round(4))

predictions.to_csv(RESULTS_DIR / 'day13_validation_predictions.csv', index=False)
performance.to_csv(RESULTS_DIR / 'day13_model_performance.csv')
feature_dictionary.to_csv(RESULTS_DIR / 'day13_feature_dictionary.csv', index=False)
fold_table.to_csv(RESULTS_DIR / 'day13_tuning_folds.csv', index=False)
tuning_summary.to_csv(RESULTS_DIR / 'day13_tuning_summary.csv', index=False)
coefficient_table.to_csv(RESULTS_DIR / 'day13_standardized_coefficients.csv')
for name, search in searches.items():
    pd.DataFrame(search.cv_results_).to_csv(RESULTS_DIR / f'day13_{name}_cv_results.csv', index=False)

analysis_input = data[['date', *base_columns]].to_csv(index=False).encode('utf-8')
configuration = {
    'target': 'Next S&P trading observation absolute 100*log price return',
    'training_outcomes_end': '2022-12-31',
    'validation_outcomes': ['2023-01-01', '2024-12-31'],
    'cv': {'n_splits': 5, 'test_size_retained_rows': 126, 'gap_retained_rows': 1},
    'tuning_metric': 'RMSE', 'ml_features': ml_features, 'day12_features': day12_features,
    'parameter_grids': {name: grid for name, (_, grid) in specifications.items()},
    'random_forest_n_estimators': 200, 'random_state': 42,
    'negative_predictions': 'Scored unchanged',
    'versions': {'python': platform.python_version(), 'numpy': np.__version__,
                 'pandas': pd.__version__, 'scikit_learn': sklearn.__version__},
    'pre2025_sp500_observation_input_sha256': hashlib.sha256(analysis_input).hexdigest(),
    'first_sp500_date': str(data['date'].min().date()),
    'ml_training_rows': len(train), 'day12_training_rows': len(day12_train),
    'validation_rows': len(validation),
}
(RESULTS_DIR / 'day13_configuration.json').write_text(json.dumps(configuration, indent=2), encoding='utf-8')
print('Results saved to:', RESULTS_DIR)
print('Figures saved to:', FIGURES_DIR)


**Your random forest achieved the lowest validation MAE and RMSE. Lasso was almost tied with it on RMSE, making it a strong simpler alternative.**

All seven methods used the same **496 forecast dates**, so this comparison is consistent. Errors are in **percentage points**.

| Method | MAE | RMSE | Bias |
|---|---:|---:|---:|
| **Random forest** | **0.4065** | **0.5180** | −0.0416 |
| Lasso | 0.4127 | 0.5182 | −0.0113 |
| Ridge | 0.4189 | 0.5218 | 0.0165 |
| Day 12 extended regression | 0.4153 | 0.5261 | −0.0325 |
| Expanded linear regression | 0.4159 | 0.5281 | −0.0332 |
| EWM | 0.4229 | 0.5301 | 0.0037 |
| Historical mean | 0.4591 | 0.5471 | 0.1425 |

**The main findings are:**

1. **Random forest improved on both benchmarks.** Compared with EWM, it reduced MAE by **3.89%** and RMSE by **2.28%**. Compared with your Day 12 regression, the reductions were **2.13%** and **1.54%**. These are modest validation gains; their statistical reliability remains untested.

2. **Lasso deserves to remain in contention.** Its RMSE was only about **0.0002 percentage points higher** than random forest’s, based on the rounded scores. Its average underprediction was also smaller: **−0.0113**, versus **−0.0416** for random forest. Forest performed better on MAE, but these results do not establish a decisive overall advantage.

3. **Adding predictors alone did not help.** Expanded ordinary linear regression performed slightly worse than the Day 12 regression on both metrics. Lasso and ridge improved on expanded linear regression, which suggests regularization helped manage the richer, overlapping predictor set. Ridge still had worse MAE than Day 12.

4. **Your tuning favored controlled complexity.** The selected forest had depth **3**, at least **10 observations per leaf**, and `max_features=0.7`. Lasso selected `alpha=0.05`; ridge selected **1000**, the largest alpha in the tested grid. That ridge result identifies the best tested value, rather than an established optimum beyond the grid.

**Your tuning and validation results answer different questions.** Ridge had the lowest mean tuning-fold RMSE (**0.7112**), followed by lasso (**0.7211**) and forest (**0.7329**). Forest then performed best during 2023–2024. Rankings can change across periods; this alone does not indicate an error. The displayed folds expand chronologically and end in 2022, consistent with your intended setup.

Your coefficient table also shows lasso removing several overlapping predictors: today’s absolute return, the 20-day mean, and both rolling standard deviations have displayed zero coefficients. **This does not establish that those variables contain no useful information**—other retained predictors may carry similar information.

The forecast plot shows all three displayed methods following broad changes in return magnitude while missing many individual spikes. Random forest appears smoother than the Day 12 regression, including around the large August 2024 movement. **None of the methods produced negative validation forecasts.**

**Keep random forest as your current validation leader, lasso as the simpler competing model, and EWM as your main benchmark.** Once you save these findings in the interpretation cell and confirm the outputs are saved, **Day 13 is complete**. Keep **2025 untouched**; the next useful analysis is to quantify uncertainty in the forecast-error differences.